<a href="https://colab.research.google.com/github/serraozkasap/ml-ai4i-predictive-maintenance/blob/main/ai4i_predictive_maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Based on CRISP-DM I'll follow these steps:

1. Business Understanding,
2. Data Understanding,
3. Data Preparation,
4. Modeling,
5. Evaluation,
6. Conclusions

## 1. Business Understanding

**Problem:** Unplanned machine stops break the production flow, increase costs and reduce efficiency. When a machine fails during production, the parts it is making often become scrap and the repair has to be done at the worst possible moment.

**My goal of this project:** I want to predict, from the sensor readings of a milling machine, whether it is going to fail (`Machine failure = 1`) and understand which conditions lead to a failure. If a failure can be predicted before it happens, maintenance can be planned, the machine can be stopped at a convenient time and scrap and maintenance costs go down.

**Which error is more expensive, I will focus for my interpration:**

- *Missed failure (false negative):* the model says "OK" but the machine fails. This causes an unplanned stop, scrap and possibly a damaged tool. **This is the expensive error. I'm planning to focus this one.**
- *False alarm (false positive):* the model says "failure" but the machine is fine. This causes one unnecessary inspection, which is a short and planned stop.

**Which metric will I focus on?**

- Only 3.39% of the records are failures (see Data Understanding session). A model that always predicts "no failure" would reach 96.61% accuracy and still miss every failure, so I understand that **accuracy is not a useful metric here**.
- My main metric is **recall on the failure class**: the share of real failures the model catches.
- A model that raises an alarm on every record has recall = 1 but is useless, so I also track **precision** (how many alarms are real failures).
- To compare models and choose the decision threshold I use the **F2-score**, which combines precision and recall but gives recall more weight, because a missed failure costs more than a false alarm.
- For every model I will report the confusion matrix, precision, recall, F2-score and the precision-recall curve (PR-AUC).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/serraozkasap/ml-ai4i-predictive-maintenance/refs/heads/main/predictive-maintenance-dataset-ai4i-2020.csv") #I just want to without uploading any file by hand (Colab deletes uploaded files when the session ends).
print(df.shape) #how many row and column we have

(10000, 14)


   # 2. Data Understanding

In [ ]:
df.head() #I want to see couple lines of dataset to understand how it looks like
#each row is one machine record with an ID (UDI, Product ID)-we can eliminate these ones on feature engineering part-, the product quality type (L/M/H),
#five sensor readings, the target "Machine failure" and five failure-mode flags (TWF, HDF, PWF, OSF, RNF)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [ ]:
df.info() #trying to see whole dataset data type and is there any null (it's important for accurate analysis)
#"Type" must be encoded as numbers before modeling
#"Product ID" is only an identifier

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [ ]:
print(df.isna().sum()) #to count the missing values per column. Missing values would have to be removed or imputed in that case we don't need imputations

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64


In [ ]:
print("Duplicated rows:", df.duplicated().sum()) #duplicated rows lower data quality. If the same row ended up in both the training set and the
#test set, the model would be tested on data it has already seen and the test score would look.
#in that case we don't need to remove, the train/test split later will not have overlapping records

Duplicated rows: 0


In [ ]:
df.describe() #I want to make plausibility check, see is the statistical metrics on our dataset like mean, std, min, quartiles and max but only for numerical columns
#it would help me to understand the range of each sensor and whether some features need scaling

#rotational speed is around 1538.78 rpm
#torque around 39.99 Nm,
#temperatures around 310 K

#all values are physically plausible so no negative values, temperatures in Kelvin around 295-314 K (make sense)

#tool wear between 0 and 253 minutes. It shows me that some of the tools didnot used before and some of them working for 253 min

#the mean of "Machine failure" is 0.0339, so only 3.39% of the records are failures. I'll mentioned it on Business Dev. part!!

#UDI only counts the rows from 1 to 10,000, so it carries no information. I'LL ELIMINATE IT ON MY DATA PREPARATION PART, NO NEED TO USE.

#distance-based and gradient-based models (KNN, logistic regression, neural networks) need scaling,
#otherwise rotational speed would dominate the other features
#tree-based models do not need scaling

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


In [ ]:
print(df["Machine failure"].value_counts())
print(df["Machine failure"].value_counts(normalize=True) * 100) #to see how the target classes are distributed (counts and percentages)

#9,661 records without failure (96.61%) and only 339 with failure (3.39%). I'LL MENTIONED THIS ON BUSINESS DEV PART!!
#The minority class is about 28 times smaller, which shows dataset imbalanced

#so accuracy would be misleading, I'll evaluate with recall, precision and F2
#I'll also use a stratified train/test split, so both sets keep the same share of failures

Machine failure
0    9661
1     339
Name: count, dtype: int64
Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64


In [ ]:
print(df["Type"].value_counts(normalize=True) * 100) #to see the distribution of the product quality types (L = low, M = medium, H = high).

#"Type" has a natural order (L < M < H), so in Data Preparation I'LL USE ORDINAL ENCODING INSTEAD OF ONE-HOT ENCODING!!

Type
L    60.00
M    29.97
H    10.03
Name: proportion, dtype: float64


In [ ]:
num_cols = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"] #I keep the five sensor columns in one list, so I can reuse it for plots and for scaling later.

print("Process temp > air temp in all rows:",(df["Process temperature [K]"] > df["Air temperature [K]"]).all()) #in there I checked process heats up the machine, so the process temperature should always
#be higher than the air temperature. A row where this is false would be a measurement error. it looks like there are no implausible temperature records.

Process temp > air temp in all rows: True


In [ ]:
modes = ["TWF", "HDF", "PWF", "OSF", "RNF"] #consistency check between the target and the five failure modes because according to the dataset description, "Machine failure" is 1 when at least one failure mode is 1.

mode_count = df[modes].sum(axis=1) #for each row: how many failure modes are active (0, 1, 2, ...)

#Case 1: a failure is recorded, but no failure mode explains it (false negative)
failure_without_mode = (df["Machine failure"] == 1) & (mode_count == 0)

# Case 2: a failure mode is recorded, but no failure (false positive)
mode_without_failure = (df["Machine failure"] == 0) & (mode_count > 0)

print("Failure but no mode:", failure_without_mode.sum()) #9 failures without a mode
print("Mode but no failure:", mode_without_failure.sum()) #18 modes without failure

#so 27 rows (0.27%) are inconsistent.these are label-quality issues. I'LL DECIDE HOW TO HANDLE THEM IN DATA PREPARATION!!!!

Failure but no mode: 9
Mode but no failure: 18


In [ ]:
print(df.loc[mode_without_failure, modes].sum()) #to find out which failure mode appears in the 18 rows where no failure was recorded.

#18 are RNF (random failure). RNF is the only mode that is not caused by the process
#it is a random event, so it cannot be predicted from the sensor readings anyway

#I think this information only supports keeping "Machine failure" as the target and treating these rows as label noise.

TWF     0
HDF     0
PWF     0
OSF     0
RNF    18
dtype: int64
